# CP6 - LangGraph investigation agent (Mode 2, D4) - Colab run

This notebook runs the CP6 agent with **free, open language models served locally by Ollama** on a free Colab GPU and
produces a small zip of result JSON files (and code-free agent traces) to hand to the next Claude Code session.

What it does, in order: clone the repository, install Go and the project, rebuild the D4 evidence on the local Colab disk
(`dataset materialize --set d4`) and **stop if it does not match the committed case list**, build the agent's commit diffs
(read-only git), put the call-graph neighbour tables in place (from the zip the build session produced, or rebuilt here with Go),
embed the D4 chunks with the fixed retriever's embedder (`bge-small`), install and start Ollama, pull the two CP5 models, check
that the server really generates text, then run the agent arms **A2** (no graph tools) and **A3** (with `callers`/`callees`) for
each model, render a preview of the results document and zip the results.

**Zero cost:** only open models run, on the Colab machine, through `http://localhost:11434/v1`. The project's endpoint rule refuses
any non-local host and every known paid API host. No paid API is called.
**What the zip contains:** result JSON (answers, stop reasons, counts, tokens, times - no prompts, no code, no raw model replies) and
the gzipped agent traces (actions, raw model replies, IDs shown, token counts - no prompts and no tool output text). The traces are
for the failure analysis in the next session; they are not committed to the repository.

Set `Runtime -> Change runtime type -> T4 GPU` first. Run the cells top to bottom; every step can be re-run safely: finished model
calls come from the response cache and finished cases from the trace directory, both on Drive, so a disconnect loses little.

**Running time per step:** not measured yet. Rough estimate from the CP5 run (about 5 s per model call on a T4) and 6-12 calls per
case: 1-2 hours per arm and model, 4-8 hours for all four runs, so expect more than one Colab session. A2 runs first for both
models (the main comparison); A3 (the ablation) after. The timings table written by the last cell is the real record; please keep it.

**Where files live:** git clones and the rebuilt evidence are on the local Colab disk (`/content/cache`, `/content/data`) because git
misbehaves on the Google Drive mount. The embedding cache, the LLM response cache, the agent traces, timings and results go on Drive.

In [ ]:
# Settings for this run. BRANCH holds the CP6 code. DRY_RUN=True swaps the language model and the embedder for deterministic fakes
# (a quick end-to-end check with no model download and no Ollama). MODELS are the same two models as CP5. ARMS: A2 = agent without
# graph tools, A3 = agent with callers/callees (A0 and A1 need no run here: A0 is recorded in every agent row, A1 comes from the
# committed CP5 results). LIMIT = None runs all 120 D4 cases; a number runs only the first N (a trial).
BRANCH = "claude/gracious-carson-ch1i9r"
REPO_URL = "https://github.com/AMAN-PRIYANSH/REPO-MIND.git"
DRY_RUN = False
GO_VERSION = "1.24.7"          # rebuilds the D4 evidence exactly as CP4/CP5 did
GRAPH_GO_VERSION = "1.26.10"   # only for rebuilding neighbour tables here: many D4 go.mod files need Go >= 1.25 (CP5b)
MODELS = ["qwen2.5-coder-7b", "llama3.1-8b"]      # aliases defined in the project's models_config
ARMS = ["A2", "A3"]
LIMIT = None
CONTEXT_TOKENS = 32768         # Ollama context length; the agent's own guard stops before 24000 prompt tokens per call

In [ ]:
# A tiny timer so the real running time of every step is recorded (written to Drive and included in the results zip).
import json, os, shutil, subprocess, sys, time
class Timings:
    def __init__(self): self.rows, self.t0, self.label = [], None, None
    def start(self, label): self.label, self.t0 = label, time.time(); print(">>>", label)
    def stop(self):
        sec = round(time.time() - self.t0, 1); self.rows.append({"step": self.label, "seconds": sec}); print(f"<<< {self.label}: {sec} s")
T = Timings()

In [ ]:
# Check whether a GPU is attached. Ollama uses it automatically; without one the 7-8B models are far too slow for the full run.
GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU available:", GPU)
if GPU:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
DEVICE = "cuda" if GPU else "cpu"
if not GPU and not DRY_RUN:
    print("WARNING: no GPU. Switch the runtime to T4 GPU, or set DRY_RUN = True for a pipeline check only.")

In [ ]:
# Mount Google Drive for the files worth keeping across a disconnect: embeddings, the LLM answer cache, agent traces, timings and
# results. The git cache and the rebuilt evidence stay on the LOCAL disk (git behaves badly on the Drive mount).
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/vulntrace_cp6"       # Drive: LLM cache, traces, results
os.makedirs(WORK, exist_ok=True)
CP5_EMB = "/content/drive/MyDrive/vulntrace_cp5/embeddings"   # the CP5 run's bge-small cache, reused if it is there
EMB = CP5_EMB if os.path.isdir(CP5_EMB) else f"{WORK}/embeddings"
CACHE = "/content/cache"                            # local disk: git clones + vulndb snapshot + Go module cache
DATA = "/content/data"                              # local disk: rebuilt D4 evidence and the agent evidence
os.makedirs(CACHE, exist_ok=True)
os.environ["VULNTRACE_CACHE"] = CACHE
SUFFIX = "_dryrun" if DRY_RUN else ""
OUT = f"{WORK}/results/cp6{SUFFIX}"
LLM_CACHE = f"{WORK}/llm_cache{SUFFIX}"
TRACES = f"{WORK}/traces{SUFFIX}"
GRAPH_ZIP = f"{WORK}/cp6_graph_tables.zip"          # optional: the neighbour tables the build session made (see the report)
DRY = "--dry-run" if DRY_RUN else ""
LIM = f"--limit {LIMIT}" if LIMIT else ""
print("Drive:", WORK, "| embeddings:", EMB, "| local git cache:", CACHE, "| local evidence:", DATA, "| results:", OUT)

In [ ]:
# Clone (or update) the repository at the CP6 branch. The code that does the work lives in the repository; this notebook only calls it.
REPO = "/content/REPO-MIND"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd /content/REPO-MIND

In [ ]:
# Install Go (Colab does not ship it). The project's small Go helper parses Go files to find function boundaries while the evidence
# is rebuilt. Installed under /content only, nothing system-wide. GRAPH_GO_VERSION is installed too but used only by the optional
# neighbour-table rebuild below.
def install_go(version):
    if not os.path.exists(f"/content/go{version}/go/bin/go"):
        subprocess.run(["wget", "-q", f"https://go.dev/dl/go{version}.linux-amd64.tar.gz", "-O", "/content/go.tgz"], check=True)
        os.makedirs(f"/content/go{version}", exist_ok=True)
        subprocess.run(["tar", "-C", f"/content/go{version}", "-xzf", "/content/go.tgz"], check=True)
    return f"/content/go{version}/go/bin"
GO_BIN, GRAPH_GO_BIN = install_go(GO_VERSION), install_go(GRAPH_GO_VERSION)
BASE_PATH = os.environ["PATH"]
os.environ["PATH"] = GO_BIN + ":" + BASE_PATH
os.environ["GOTOOLCHAIN"] = "local"
os.environ["CGO_ENABLED"] = "0"
print(subprocess.run(["go", "version"], capture_output=True, text=True).stdout)

In [ ]:
# Install this project with the [ml] extra (sentence-transformers for the bge-small embedder; Colab already has torch with CUDA)
# and the [agent] extra (LangGraph, pinned in pyproject.toml).
!pip install -q -e ".[ml,agent]"
# The notebook's Python session does not see the new install until it is restarted, so point it at the checkout directly.
import sys, importlib
sys.path.insert(0, "/content/REPO-MIND/src")
importlib.invalidate_caches()
!python -m vulntrace doctor

In [ ]:
# Fetch the pinned golang/vulndb snapshot and the Go helper's dependencies (both cached; quick on re-runs).
T.start("fetch vulndb + build helper")
!python -m vulntrace dataset fetch
!python -m vulntrace helper fetch
!python -m vulntrace helper build
T.stop()

In [ ]:
# Rebuild the D4 evidence (release ranges and candidate code) for the 120 committed case IDs, on the local disk. Slow and
# network-heavy: sequential git fetches with a pause between them. The CP5 run took about 28 minutes.
T.start("materialize D4")
!python -u -m vulntrace dataset materialize --set d4 --data-dir "{DATA}" --summary-out "{WORK}/D4_materialize.json"
T.stop()

In [ ]:
# GATE: stop here if the rebuilt evidence does not match the committed case list (more than 3 cases with a different status, or no
# candidate functions at all). If this cell stops the notebook, do NOT continue: re-run the cell above (a mismatching case is
# rebuilt) and read the git errors it prints, or run  !python -m vulntrace dataset diagnose --case GO-2020-0038
from collections import Counter
from vulntrace.eval import evidence_gate
case_list = json.load(open("docs/results/datasets/D4_cases.json"))
statuses, n_candidates = evidence_gate.evidence_statuses(case_list, DATA, "release_range")
problems = evidence_gate.check_evidence(case_list, statuses, n_candidates)
print("D4 materialised statuses:", dict(Counter(statuses.values())), "| expected:", case_list.get("status_counts"), "| candidates:", n_candidates)
if problems:
    raise SystemExit("STOP - the evidence is not usable: " + "; ".join(problems))
print("gate passed")

In [ ]:
# Agent evidence 1: the Go diff of every commit of every release range (for the show_commit tool), read from the local git cache with
# `git show` (read-only; nothing is executed). About 2 minutes. Compared with the build session's content hashes below.
T.start("agent prepare --diffs")
!python -u -m vulntrace agent prepare --set d4 --diffs --data-dir "{DATA}" --summary-out "{WORK}/agent_evidence_colab.json"
T.stop()

In [ ]:
# Agent evidence 2: the call-graph neighbour tables for arm A3 (one-hop callers and callees of every candidate). Preferred: unzip the
# tables the build session made (GRAPH_ZIP on Drive; see the CP6 report for how to get it). Otherwise rebuild them here with Go
# GRAPH_GO_VERSION and the Go module proxy - CPU-heavy, about an hour or more on Colab's 2 CPUs. Either way they are compared with the
# committed summary docs/results/cp6/agent_evidence.json; a case without a table runs A3 without the graph tools and is counted.
import zipfile
from vulntrace.agent import prepare as agent_prepare
if os.path.exists(GRAPH_ZIP):
    with zipfile.ZipFile(GRAPH_ZIP) as z:
        for name in z.namelist():
            if not name.startswith("agent/graph/") or not name.endswith(".json.gz") or ".." in name:
                raise SystemExit(f"STOP - unexpected file in {GRAPH_ZIP}: {name}")
        z.extractall(DATA)
    print("neighbour tables unpacked from", GRAPH_ZIP)
else:
    T.start("agent prepare --graph (rebuild in Colab)")
    os.environ["PATH"] = GRAPH_GO_BIN + ":" + BASE_PATH
    !python -m vulntrace helper build
    !python -u -m vulntrace agent prepare --set d4 --graph --jobs 2 --data-dir "{DATA}" --gomodcache /content/cache/gomod --summary-out "{WORK}/agent_evidence_colab.json"
    os.environ["PATH"] = GO_BIN + ":" + BASE_PATH
    T.stop()
committed = json.load(open("docs/results/cp6/agent_evidence.json"))
cmp = agent_prepare.compare_with_summary([c["id"] for c in case_list["cases"]], DATA, committed)
for kind, res in cmp.items():
    print(kind, {k: len(v) for k, v in res.items()}, "| different:", res["different"][:10])

In [ ]:
# Embed every D4 chunk (code+commits variant, the one the fixed retriever uses) with bge-small, caching vectors on Drive by the chunk
# text's hash (the CP5 cache is reused when present, so this is quick). Skipped in a dry run.
if not DRY_RUN:
    T.start("embed D4 chunks (bge-small)")
    !python -u -m vulntrace index build --set d4 --embedder bge-small --variant "code+commits" --data-dir "{DATA}" --embedding-cache "{EMB}" --device {DEVICE}
    T.stop()

In [ ]:
# Install and start Ollama on this machine (listening on 127.0.0.1 only) with a 32k-token context, then wait until it answers.
# Skipped in a dry run. zstd is needed by Ollama's installer. `which ollama` only proves the CLI exists (a partial install has no
# runner and fails every generate call with HTTP 500), so the pull cell below also makes one tiny generate call and, if that fails,
# calls reinstall_ollama() once before the run goes on.
import urllib.request
def ollama_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3).read(); return True
    except Exception:
        return False
def start_ollama():
    env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(CONTEXT_TOKENS))
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up(): break
        time.sleep(2)
def reinstall_ollama():
    subprocess.run("pkill -f 'ollama serve'", shell=True); time.sleep(2)
    subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    start_ollama()
if not DRY_RUN:
    T.start("install + start Ollama")
    if shutil.which("ollama") is None:
        !apt-get -qq install -y zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh
    if not ollama_up():
        start_ollama()
    print("ollama answering:", ollama_up())
    print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout)
    T.stop()

In [ ]:
# Pull the models and print the exact tag and digest the server reports. A model that cannot be pulled is recorded in model_problems
# and skipped - there is no fallback to any other service. Then the install check: one tiny generate call with the first model that
# pulled; on failure reinstall once; if it still fails the notebook stops here (OllamaSetupError) instead of failing call by call.
from vulntrace.llm import models_config
model_problems, MODEL_INFO = {}, {}
if not DRY_RUN:
    for alias in MODELS:
        tag = models_config.resolve(alias)["ollama_tag"]
        T.start(f"pull {tag}")
        r = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)
        T.stop()
        if r.returncode != 0:
            model_problems[alias] = (r.stderr or r.stdout)[-500:]; print("PULL FAILED:", alias, model_problems[alias])
    from vulntrace.llm import ollama_setup
    pulled = [a for a in MODELS if a not in model_problems]
    if pulled:
        probe_tag = models_config.resolve(pulled[0])["ollama_tag"]
        ollama_setup.ensure_working(lambda: ollama_setup.probe_generate(probe_tag), reinstall_ollama)
    tags = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]
    MODEL_INFO = {m["name"]: {"digest": m["digest"], "size": m["size"], "details": m.get("details")} for m in tags}
    print(json.dumps(MODEL_INFO, indent=1))
MODELS_TO_RUN = ["fake"] if DRY_RUN else [m for m in MODELS if m not in model_problems]
print("models to run:", MODELS_TO_RUN, "| problems:", model_problems or "none")

In [ ]:
# The agent runs: for each arm (A2 first, the main comparison) and each model, the agent investigates every D4 case with at most 8
# steps and submits a ranked answer of 1-3 candidate IDs (or abstains). Results: OUT/agent__<arm>__<model>.json; one gzipped trace per
# case in TRACES. A re-run resumes: cases with a finished trace for the same configuration are not run again. Three cases in a row with
# a server error stop the run (nothing is written for that arm; re-run the cell once the server is up).
for arm in ARMS:
    for m in MODELS_TO_RUN:
        T.start(f"agent {arm} {m}")
        !python -u -m vulntrace eval symbols --strategy agent --set d4 --arm {arm} --model {m} {DRY} {LIM} --data-dir "{DATA}" --embedding-cache "{EMB}" --device {DEVICE} --llm-cache "{LLM_CACHE}" --trace-dir "{TRACES}" --out "{OUT}"
        T.stop()

In [ ]:
# Render the results document from the files just written, to look at it here. The next Claude Code session renders the committed copy
# the same way from the zip (with the CP5 S1 files for arm A1, which are in the repository).
!python -m vulntrace results render --doc cp6 --input "{OUT}" --out "{WORK}/CP6_AGENT_RESULTS_preview.md"
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK}/CP6_AGENT_RESULTS_preview.md").read()[:12000]))

In [ ]:
# Zip the result JSON (agent files, evidence summaries, timings, model digests, model problems) and the gzipped traces, and download it.
# No prompts, no code, no models and no embeddings go into the zip. Refuses to build a zip if an arm produced no result file (an empty
# run must never be handed over as a result).
import zipfile, glob
os.makedirs(OUT, exist_ok=True)
json.dump(model_problems, open(f"{OUT}/model_problems.json", "w"), indent=1)
json.dump({"models": MODEL_INFO, "timings": T.rows, "dry_run": DRY_RUN, "gpu": GPU, "context_tokens": CONTEXT_TOKENS,
           "limit": LIMIT, "graph_tables_from_zip": os.path.exists(GRAPH_ZIP), "evidence_vs_committed":
           {k: {kk: len(vv) for kk, vv in v.items()} for k, v in cmp.items()}},
          open(f"{OUT}/colab_run.json", "w"), indent=1)
expected = [f"{OUT}/agent__{arm}__{models_config.resolve(m)['alias']}.json" for arm in ARMS for m in MODELS_TO_RUN]
missing = [p for p in expected if not os.path.exists(p)]
if missing:
    raise SystemExit("STOP - these result files are missing, not building a zip: " + ", ".join(missing))
zpath = f"{WORK}/cp6_results{SUFFIX}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(f"{OUT}/*.json")):
        z.write(p, "cp6/" + os.path.basename(p))
    for n in ("D4_materialize.json", "agent_evidence_colab.json"):
        if os.path.exists(f"{WORK}/{n}"): z.write(f"{WORK}/{n}", f"cp6/{n.replace('D4_materialize.json', 'D4_materialize_colab.json')}")
    for p in sorted(glob.glob(f"{TRACES}/*/*.json.gz")):
        z.write(p, "traces/" + os.path.relpath(p, TRACES))
print(zpath, os.path.getsize(zpath) // 1024, "KiB")
from google.colab import files
files.download(zpath)